In [ ]:
from collections import OrderedDict

import pandas as pd
import plotly.express as px

### Questão central de negócio

> quais fatores operacionais realmente influenciam a satisfação do cliente e como a empresa pode agir de forma proativa para melhorar a experiência antes mesmo da aplicação da pesquisa de NPS?

### Entendimento do negócio: 

nessa primeira etapa, queremos exercitar o seu pensamento analítico, não código. Nos traga a resposta para as seguintes perguntas de negócio:

- [ ] Qual problema de negócio está sendo resolvido?
- [ ] Por que o NPS é importante para um e-commerce?
- [ ] Quais áreas poderiam se beneficiar desses insights? Exemplos: logística, atendimento, pricing, produto etc.

In [2]:
## Config Pandas
pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../data/desafio_nps_fase_1.csv")
metadados_colunas = pd.read_json("../data/metadados_desafio_nps_fase_1.json")

### Criando uma visão geral do dataset com a documentação das colunas dos dados

In [3]:
# Separar variáveis quantitativas e qualitativas
cols_quant = [col for col, meta in metadados_colunas.items() if meta["tipo"] == "quantitativa"]
cols_qual = [col for col, meta in metadados_colunas.items() if meta["tipo"] == "qualitativa"]

# Calcula as estatísticas descritivas APENAS para as quantitativas
df_infos_quant = df[cols_quant].describe().T

# Adiciona metadados às quantitativas
df_infos_quant["descricao"] = df_infos_quant.index.map(
    lambda col: metadados_colunas.get(col, {}).get("descricao", "—")
)
df_infos_quant["grupo"] = df_infos_quant.index.map(
    lambda col: metadados_colunas.get(col, {}).get("grupo", "—")
)
df_infos_quant["tipo"] = df_infos_quant.index.map(
    lambda col: metadados_colunas.get(col, {}).get("tipo", "—")
)
df_infos_quant["natureza"] = df_infos_quant.index.map(
    lambda col: metadados_colunas.get(col, {}).get("natureza", "—")
)
# Adiciona a informação de linhas vazias por coluna
df_infos_quant["nulos"] = df_infos_quant.index.map(lambda col: df[col].isnull().sum())
# Adiciona o tipo de dado real da coluna
df_infos_quant["dtype"] = df_infos_quant.index.map(lambda col: df[col].dtype)
# Adiciona o count calculado para todas as variáveis (embora já esteja no describe, garantimos a coluna)
df_infos_quant["count"] = df_infos_quant.index.map(lambda col: df[col].count())

cols_novas_quant = ["descricao", "grupo", "tipo", "natureza", "dtype", "nulos", "count"] + [
    col
    for col in df_infos_quant.columns
    if col not in ["descricao", "grupo", "tipo", "natureza", "dtype", "nulos", "count"]
]
df_infos_quant = df_infos_quant[cols_novas_quant]

# Para qualitativas, mostrar os metadados, tipo de dado e count
df_infos_qual = pd.DataFrame(
    OrderedDict(
        [
            (
                col,
                {
                    "descricao": metadados_colunas[col]["descricao"],
                    "grupo": metadados_colunas[col]["grupo"],
                    "tipo": metadados_colunas[col]["tipo"],
                    "natureza": metadados_colunas[col]["natureza"],
                    "dtype": df[col].dtype,
                    "nulos": df[col].isnull().sum(),
                    "count": df[col].count(),
                },
            )
            for col in cols_qual
        ]
    )
).T

cols_novas_qual = ["descricao", "grupo", "tipo", "natureza", "dtype", "nulos", "count"]
df_infos_qual = df_infos_qual[cols_novas_qual]

# Concatena: primeiro quantitativas, depois qualitativas (ordenadas pelo tipo)
df_infos = pd.concat([df_infos_quant, df_infos_qual], axis=0)

# Exibe resultado ordenado por tipo, depois pela natureza
df_infos = df_infos.sort_values(by=["tipo", "natureza"], ascending=[True, False]).fillna("—")

df_infos

,descricao,grupo,tipo,natureza,dtype,nulos,count,mean,std,min,25%,50%,75%,max
nps_score,"Net Promoter Score após a experiência de compra. Escala 0–10 (detratores: 0–6, neutros: 7–8, promotores: 9–10).",Satisfação,qualitativa,ordinal,float64,0,2500,—,—,—,—,—,—,—
csat_internal_score,Customer Satisfaction Score interno pós-atendimento/entrega. Escala 0–10.,Satisfação,qualitativa,ordinal,float64,0,2500,—,—,—,—,—,—,—
customer_id,Identificador único do cliente.,Cliente,qualitativa,nominal,int64,0,2500,—,—,—,—,—,—,—
customer_region,"Região geográfica do cliente no Brasil (Norte, Nordeste, Centro-Oeste, Sudeste, Sul).",Cliente,qualitativa,nominal,str,0,2500,—,—,—,—,—,—,—
order_id,Identificador único do pedido.,Pedido,qualitativa,nominal,int64,0,2500,—,—,—,—,—,—,—
customer_age,Idade do cliente em anos.,Cliente,quantitativa,discreta,int64,0,2500,43.396,14.888487,18.0,31.0,43.0,56.0,69.0
customer_tenure_months,"Tempo de relacionamento do cliente com a plataforma, em meses.",Cliente,quantitativa,discreta,int64,0,2500,61.3224,34.478729,1.0,31.0,62.0,91.0,119.0
items_quantity,Quantidade de itens incluídos no pedido.,Pedido,quantitativa,discreta,int64,0,2500,3.4708,1.687331,1.0,2.0,3.0,5.0,6.0
payment_installments,Número de parcelas escolhidas pelo cliente no pagamento.,Pedido,quantitativa,discreta,int64,0,2500,6.004,3.159743,1.0,3.0,6.0,9.0,11.0
delivery_time_days,"Tempo total de entrega do pedido, em dias corridos.",Logística,quantitativa,discreta,int64,0,2500,8.022,3.770411,2.0,5.0,8.0,11.0,14.0


In [15]:
df_infos.sort_values("grupo")

,descricao,grupo,tipo,natureza,dtype,nulos,count,mean,std,min,25%,50%,75%,max
complaints_count,Número de reclamações formais registradas pelo cliente.,Atendimento,quantitativa,discreta,int64,0,2500,4.1504,1.784223,0.0,3.0,4.0,5.0,11.0
resolution_time_days,Tempo em dias para resolução do problema reportado pelo cliente.,Atendimento,quantitativa,discreta,int64,0,2500,5.4856,3.458002,0.0,2.0,6.0,8.0,11.0
customer_service_contacts,Número de contatos do cliente com o SAC referentes ao pedido.,Atendimento,quantitativa,discreta,int64,0,2500,1.5196,1.231512,0.0,1.0,1.0,2.0,7.0
customer_id,Identificador único do cliente.,Cliente,qualitativa,nominal,int64,0,2500,—,—,—,—,—,—,—
customer_region,"Região geográfica do cliente no Brasil (Norte, Nordeste, Centro-Oeste, Sudeste, Sul).",Cliente,qualitativa,nominal,str,0,2500,—,—,—,—,—,—,—
customer_age,Idade do cliente em anos.,Cliente,quantitativa,discreta,int64,0,2500,43.396,14.888487,18.0,31.0,43.0,56.0,69.0
customer_tenure_months,"Tempo de relacionamento do cliente com a plataforma, em meses.",Cliente,quantitativa,discreta,int64,0,2500,61.3224,34.478729,1.0,31.0,62.0,91.0,119.0
delivery_time_days,"Tempo total de entrega do pedido, em dias corridos.",Logística,quantitativa,discreta,int64,0,2500,8.022,3.770411,2.0,5.0,8.0,11.0,14.0
delivery_attempts,Número de tentativas do transportador para concluir a entrega.,Logística,quantitativa,discreta,int64,0,2500,2.0056,0.815497,1.0,1.0,2.0,3.0,3.0
delivery_delay_days,Número de dias de atraso em relação à data prometida. Zero indica entrega no prazo.,Logística,quantitativa,discreta,int64,0,2500,2.1872,1.454442,0.0,1.0,2.0,3.0,8.0


## Metodologia: Análise Baseada na Jornada do Cliente

Para que esta Análise Exploratória (EDA) reflita a realidade do negócio, estruturamos a investigação seguindo o **ciclo de vida cronológico** da experiência do cliente no e-commerce. O objetivo é isolar em qual etapa do fluxo a experiência começa a se degradar, identificando os pontos de ruptura que impactam o NPS final.

A análise seguirá quatro etapas consecutivas:

1. **O Momento da Compra (Expectativa):** Análise do impacto financeiro inicial (`order_value`, `discount_value`, `payment_installments`).
2. **A Logística e Entrega (A Espera):** Investigação do limite de tolerância do cliente com prazos e falhas (`delivery_time_days`, `delivery_delay_days`, `delivery_attempts`).
3. **O Atendimento (O SOS):** Avaliação da capacidade do suporte em reverter atritos operacionais (`customer_service_contacts`, `resolution_time_days`).
4. **O Desfecho (Retenção):** Cruzamento da percepção final (`nps_score`) com o comportamento real de recompra de 30 dias (`repeat_purchase_30d`).

> **Premissa de Negócio:** Mapear a jornada de forma cronológica nos permite transformar dados operacionais frios em uma história visual e acionável para tomada de decisão.

### 📊 Análise por Grupo: Operação vs. Alvo (Satisfação)

Para identificar quais fatores operacionais mais impactam o nosso alvo, dividiremos as análises cruzando cada grupo de dados diretamente com a satisfação do cliente.

| Grupo Analisado | Variáveis Operacionais | Alvo de Negócio |
| :--- | :--- | :--- |
| **1. Pedido** | `order_value`, `discount_value`, `payment_installments`... | `nps_score` / Categoria |
| **2. Logística** | `delivery_time_days`, `delivery_delay_days`, `delivery_attempts`... | `nps_score` / Categoria |
| **3. Atendimento**| `customer_service_contacts`, `resolution_time_days`... | `nps_score` / Categoria |
| **4. Cliente** | `customer_age`, `customer_region`, `customer_tenure_months`... | `nps_score` / Categoria |

In [16]:
cols_heatmap = [
    "payment_installments",
    "items_quantity",
    "order_value",
    "discount_value",
    "nps_score",
]
corr_matrix = df[cols_heatmap].corr()

fig = px.imshow(
    corr_matrix,
    title="Correlação: Variáveis do Pedido vs NPS Score",
    color_continuous_scale="RdBu_r",
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    labels=dict(color="Correlação"),
)
fig.update_layout(
    width=700,
    height=600,
    title_x=0.5,
)
fig.show()